# 4. Datareduktion
PCA, t-SNE och UMAP (extra) på WHR26-features.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

df = pd.read_csv("../data/cleaned_data.csv")

faktorer = [
    "Explained by: Log GDP per capita",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Generosity",
    "Explained by: Perceptions of corruption"
]

faktor_etiketter = ["BNP", "Socialt stöd", "Hälsa", "Frihet", "Generositet", "Korruption"]

df_2019 = df[df["Year"] == 2019].set_index("Country name")[["Life evaluation (3-year average)"]]
df_2025 = df[df["Year"] == 2025].set_index("Country name")[["Life evaluation (3-year average)"]]
delta = (df_2025 - df_2019).rename(columns={"Life evaluation (3-year average)": "delta"})

df_X = df[df["Year"] == 2025].set_index("Country name")[faktorer]
df_X = df_X.join(delta).dropna()

X = df_X[faktorer].values
länder = df_X.index.tolist()
färger = df_X["delta"].values

print(f"X: {X.shape}")

In [ ]:
# Skala om alla 6 faktorer så att varje kolumn får medelvärde 0 och standardavvikelse 1.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Bekräfta: varje kolumn ska ha medelvärde ≈0 och std ≈1
print("Medelvärden:", X_scaled.mean(axis=0).round(3))
print("Std:        ", X_scaled.std(axis=0).round(3))


# Till presentation:  BNP har medelvärde ~1.27, Korruption ~0.15. 
# Utan skalning tror PCA att BNP är "viktigare" bara för att den har större tal. det är inte sant.

In [ ]:
# Kör PCA på alla 6 dimensioner och titta på hur mycket varians varje principalkomponent "(PC)" förklarar.
pca_full = PCA(n_components=6)
pca_full.fit(X_scaled)

varians = pca_full.explained_variance_ratio_ * 100
kumulativ = np.cumsum(varians)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"PC{i}" for i in range(1, 7)],
    y=varians,
    name="Per komponent",
    marker_color="#4fc3f7",
    text=[f"{v:.1f}%" for v in varians],
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_trace(go.Scatter(
    x=[f"PC{i}" for i in range(1, 7)],
    y=kumulativ,
    name="Kumulativt",
    mode="lines+markers",
    line=dict(color="#ff7043", width=2.5),
    marker=dict(size=8)
))

fig.add_hline(y=80, line_dash="dash", line_color="#888888",
              annotation_text="80%", annotation_font_color="white")

fig.update_layout(
    title=dict(text="Scree plot – förklarad varians per principalkomponent",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="Principalkomponent", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(title="Förklarad varians (%)", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    legend=dict(font=dict(color="white")),
    height=450
)

fig.show()
    
# Till presentation: Scree-plotten visar om 2 komponenter räcker för att fånga mönstret. 
# Om PC1+PC2 förklarar t.ex. 70% av variansen är en 2D-plot meningsfull.

In [ ]:
# Reducera till 2 dimensioner och plotta varje land som en punkt. Färgkoda efter delta (grön = vinnare, röd = förlorare).
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    mode="markers",
    marker=dict(
        size=8,
        color=färger,
        colorscale="RdYlGn",
        colorbar=dict(title="Delta", tickfont=dict(color="white"),
                      title_font=dict(color="white")),
        showscale=True
    ),
    text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
    hovertemplate="%{text}<extra></extra>"
))

fig.update_layout(
    title=dict(text="PCA – länder i 2D-rum (färg = förändring 2019→2025)",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title=f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% varians)",
               title_font=dict(color="white"), tickfont=dict(color="white"),
               gridcolor="#444444"),
    yaxis=dict(title=f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% varians)",
               title_font=dict(color="white"), tickfont=dict(color="white"),
               gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=600
)

fig.show()

# Till presentation: Det är den centrala visualiseringen. visar om vinnare och förlorare grupperar sig i PCA-rummet.

In [ ]:
# Titta på PCA loadings. hur mycket varje ursprunglig faktor bidrar till PC1 och PC2?
# Visa hur mycket varje faktor bidrar till PC1 och PC2.
loadings = pd.DataFrame(
    pca2.components_.T,
    index=faktor_etiketter,
    columns=["PC1", "PC2"]
)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=loadings.index,
    x=loadings["PC1"],
    orientation="h",
    name="PC1",
    marker_color="#4fc3f7",
    text=loadings["PC1"].round(2),
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_trace(go.Bar(
    y=loadings.index,
    x=loadings["PC2"],
    orientation="h",
    name="PC2",
    marker_color="#ff7043",
    text=loadings["PC2"].round(2),
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_vline(x=0, line_color="white", line_width=0.8)

fig.update_layout(
    title=dict(text="PCA loadings – faktorernas bidrag till PC1 och PC2",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="Loading", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(tickfont=dict(color="white")),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    legend=dict(font=dict(color="white")),
    barmode="group",
    height=400
)

fig.show()

# Till presentation: Det svarar på bifråga 2. "vilken faktor skiljer vinnare från förlorare?" 
# PC1 representerar den dominerande variationsriktningen; den faktor med högst loading där är den viktigaste.

In [ ]:
# Kör UMAP på samma X_scaled och plotta på samma sätt som PCA.
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    mode="markers",
    marker=dict(
        size=8,
        color=färger,
        colorscale="RdYlGn",
        colorbar=dict(title="Delta", tickfont=dict(color="white"),
                      title_font=dict(color="white")),
        showscale=True
    ),
    text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
    hovertemplate="%{text}<extra></extra>"
))

fig.update_layout(
    title=dict(text="UMAP – länder i 2D-rum (färg = förändring 2019→2025)",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="UMAP-1", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(title="UMAP-2", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=600
)

fig.show()

# Till presentation: UMAP är bättre på att bevara lokala klusterstrukturer. 
# Jämförelsen mot PCA visar om det finns icke-linjära mönster som PCA missar.

In [ ]:
# Kör UMAP tre gånger med olika n_neighbors och visa plottarna sida vid sida.
# n_neighbors styr hur "lokal" UMAP är. Lågt värde = mer fokus på närmaste grannar, högt värde = mer global struktur.
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["n_neighbors=5", "n_neighbors=15", "n_neighbors=30"])

for col, n in enumerate([5, 15, 30], 1):
    emb = umap.UMAP(n_neighbors=n, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    fig.add_trace(go.Scatter(
        x=emb[:, 0], y=emb[:, 1],
        mode="markers",
        marker=dict(size=6, color=färger, colorscale="RdYlGn", showscale=(col == 3),
                    colorbar=dict(title="Delta", tickfont=dict(color="white"),
                                  title_font=dict(color="white"))),
        text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title=dict(text="UMAP – hyperparametertest (n_neighbors)",
               font=dict(color="white", size=16), x=0.5),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=500
)

for ann in fig.layout.annotations:
    ann.font.color = "white"

fig.update_xaxes(gridcolor="#444444", tickfont=dict(color="white"))
fig.update_yaxes(gridcolor="#444444", tickfont=dict(color="white"))

fig.show()

# Till presentation: n_neighbors styr balansen mellan lokal och global struktur. 
# Arbetsplanen kräver att ni testar 5, 15 och 30 och jämför. det visar att ni förstår algoritmen.
# n_neighbors=5 → tight lokala kluster. n_neighbors=30 → mer global struktur, liknar PCA mer.